# Plantilla de Anotación Manual RSL
Convierte tus anotaciones manuales (lo que ya estás haciendo) en ejemplos de entrenamiento.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pypdf', 'openpyxl'])
print('OK')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# AQUI VIVES TU — agrega cada articulo que ya anotaste manualmente
# Estructura: nombre_pdf, dominio, decision, y un dict con columna->valor
# Si una columna NO aplica al articulo -> valor empieza con [SIN INFORMACION]
# ════════════════════════════════════════════════════════════════════

ANOTACIONES = [

    # ── INGENIERIA ──────────────────────────────────────────────────
    {
        'pdf':      'A Statistical Framework of Watermarks for Large Language Models.pdf',
        'dominio':  'ingenieria',
        'decision': 'excluir',
        'columnas': {
            'Tipo de estudio':
                'Marco estadistico teorico y evaluacion empirica centrada en '
                'deteccion de marcas de agua en texto generado por LLMs.',

            'Herramientas comparadas':
                'Se comparan reglas estadisticas y algoritmos de decodificacion '
                'de marcas de agua (Gumbel-max y transformada inversa). '
                'No se comparan herramientas SAST ni LLMs para analisis de codigo.',

            'Reduccion de falsos positivos':
                '[SIN INFORMACION] El estudio controla la tasa de falsos positivos '
                'pero referida a detectar texto humano como generado por IA, '
                'no a vulnerabilidades de software.',

            'Conjunto de codigo analizado':
                '[SIN INFORMACION] No se analiza codigo fuente. Se uso el dataset '
                'C4 (noticias) y salidas de ChatGPT, OPT-1.3B y Sheared-LLaMA.',

            'Resultados principales':
                'Se derivan reglas matematicas optimas que mejoran la eficiencia '
                'estadistica para detectar marcas de agua invisibles en textos de IA.',

            'Limitaciones del estudio':
                'Falta de conocimiento sobre distribuciones exactas de tokens del LLM '
                'y cuestiones criptograficas (colisiones de hash) al generar texto.',
        }
    },

    # ── AGREGA AQUI EL SIGUIENTE ARTICULO ───────────────────────────
    # {
    #     'pdf':      'nombre_del_archivo.pdf',
    #     'dominio':  'ingenieria',   # o 'medicina' o 'educacion'
    #     'decision': 'incluir',      # o 'excluir'
    #     'columnas': {
    #         'Tipo de estudio':            'texto que escribiste...',
    #         'Herramientas comparadas':    'texto que escribiste...',
    #         'Reduccion de falsos positivos': '[SIN INFORMACION] porque...',
    #         'Conjunto de codigo analizado':  'texto...',
    #         'Resultados principales':        'texto...',
    #         'Limitaciones del estudio':      'texto...',
    #     }
    # },

]

print(f'Articulos anotados: {len(ANOTACIONES)}')
for a in ANOTACIONES:
    print(f"  {a['dominio']:12} | {a['decision']:8} | {a['pdf'][:60]}")

In [ ]:
import json, re
from pypdf import PdfReader
from pathlib import Path

BASE_DIR = '/content'

def extraer_chunk_relevante(pdf_path: str, columna: str, valor: str) -> str:
    """
    Busca en el PDF el parrafo que mejor corresponde al valor anotado.
    Si no encuentra nada bueno, devuelve un fragmento representativo del articulo.
    """
    try:
        reader  = PdfReader(pdf_path)
        texto   = ' '.join(
            page.extract_text() or '' for page in reader.pages
        )
    except:
        return valor[:500]  # fallback: usar el valor mismo como chunk

    # Dividir en oraciones y buscar la mas cercana al valor anotado
    palabras_valor = set(re.findall(r'\w+', valor.lower()))
    palabras_valor -= {'el','la','los','las','de','del','en','un','una',
                       'y','o','a','con','por','para','se','no','es','son',
                       'sin','informacion','the','a','an','of','in','to','and'}

    # Partir texto en ventanas de ~400 palabras
    tokens   = texto.split()
    ventanas = [' '.join(tokens[i:i+400]) for i in range(0, len(tokens)-200, 200)]

    if not ventanas:
        return texto[:600]

    # Escoger la ventana con mayor solapamiento con el valor anotado
    mejor = max(ventanas,
        key=lambda v: len(palabras_valor & set(re.findall(r'\w+', v.lower()))))

    return mejor[:800]

print('Funcion de extraccion lista')

In [ ]:
# ─── CONVERTIR ANOTACIONES A EJEMPLOS DE ENTRENAMIENTO ────────────────────────
dataset = []

for anotacion in ANOTACIONES:
    pdf_name = anotacion['pdf']
    dominio  = anotacion['dominio']
    pdf_path = str(Path(BASE_DIR) / dominio / pdf_name)

    print(f"Procesando: {pdf_name[:60]}")

    for columna, valor in anotacion['columnas'].items():
        # Buscar el chunk del PDF que mejor corresponde a este valor
        chunk = extraer_chunk_relevante(pdf_path, columna, valor)

        dataset.append({
            'chunk':      chunk,
            'column':     columna,
            'output':     valor,
            'dominio':    dominio,
            'decision':   anotacion['decision'],
            'fuente':     pdf_name,
        })

pos = sum(1 for x in dataset if '[SIN INFORMACION]' not in x['output'])
neg = len(dataset) - pos
print(f'\nTotal ejemplos  : {len(dataset)}')
print(f'Con informacion : {pos}')
print(f'Sin informacion : {neg}')
print(f'Ratio           : {pos/max(1,neg):.1f}:1')

In [ ]:
# ─── GUARDAR ──────────────────────────────────────────────────────────────────
OUTPUT_JSON = '/content/dataset_anotado.json'
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)
print(f'Guardado: {OUTPUT_JSON}')

# Generar bloque RAW listo para pegar en el notebook de entrenamiento
def esc(s):
    return s.replace('\\','').replace("'",'\u2019').replace('\n',' ').replace('\r','').strip()

lineas = ['RAW = [']
for ej in dataset:
    lineas.append(f"    {{'chunk': '{esc(ej['chunk'][:700])}',")
    lineas.append(f"     'column': '{esc(ej['column'])}',")
    lineas.append(f"     'output': '{esc(ej['output'])}'}},")
lineas.append(']')

RAW_FILE = '/content/raw_entrenamiento.py'
with open(RAW_FILE, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lineas))
print(f'RAW generado: {RAW_FILE}')

from google.colab import files
files.download(OUTPUT_JSON)
files.download(RAW_FILE)